# Building a Debugging Assistant

Learning objective: Build a local chatbot that teaches structured debugging, verification, and reflection.

The model does not give final corrected code. It helps you build a debugging habit:

1. **Observe** — What error or output do I see?
2. **Root Cause** — What might be causing the bug?
3. **Check** — What should I inspect next?
4. **Verify** — Did my fix actually work?
5. **Reflect** — What will I check next time?


![Guided debugging workflow](debugging_workflow.png)

The main idea is **verification before trust**. The model can suggest a root cause, but Python runs the check and you verify the result.


## Why this still matters

AI can write code quickly. That is useful.

But speed is not the same as correctness.

| AI can help with... | You still need to check... |
|---|---|
| 📝 Writing a first draft | Did it solve the right problem? |
| 🐛 Explaining a possible bug | Is the explanation supported by evidence? |
| 🔧 Suggesting a fix | Did we test the fix? |
| 🖼️ Reading an image or report | Did it extract the details correctly? |
| 🧰 Calling a tool | Did the program run the tool safely? |

In this notebook, you will see a real example: the base output may give corrected code even when the prompt asks it not to.

**Goal:** learn how to say, "This answer looks helpful, but I know how to check it."


## Step 1: Imports

Run the setup imports first. These libraries load the model, read JSON, run small checks, and display simple widgets.


In [19]:
# JUST RUN THIS CELL
from llama_cpp import Llama  # loads local GGUF models
import os  # works with file paths
import json  # reads JSON text as Python data
import subprocess  # runs a small diagnostic program
import sys  # finds the current Python program
import base64  # encodes an image for the API
import time  # measures API call time
import pandas as pd  # makes tables
from IPython.display import display  # displays widgets
import ipywidgets as widgets  # creates dropdown menus

try:
    from dotenv import load_dotenv  # reads API keys from a .env file
except Exception as error:
    print("python-dotenv is not installed:", error)

try:
    from openai import OpenAI  # OpenRouter uses the OpenAI client format
except Exception as error:
    print("openai is not installed:", error)


## Step 2: Load the local model

This notebook uses **Qwen2.5 3B Instruct Q4_K_M** in GGUF format.

GGUF is a local model file format used by llama.cpp tools.


In [20]:
# RUN THIS CELL
model_directory = "/Users/seohachoi/models"
local_model_name = "Qwen2.5-3B-Instruct-Q4_K_M.gguf"
official_model_name = "qwen2.5-3b-instruct-q4_k_m.gguf"

model_path = os.path.join(model_directory, local_model_name)

if not os.path.exists(model_path):
    model_path = os.path.join(model_directory, official_model_name)

model = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=None,
    verbose=False,
)

print("Model loaded:", model_path)


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded: /Users/seohachoi/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf


## Step 3: Get base output

First, ask the model for debugging help with only a light prompt. This shows what can happen before we add stricter instructions.


In [21]:
# RUN THIS CELL
base_prompt = """
My Python code does not work. Please help me debug it, but do not give me the solution or corrected code.

Problem:
I want to write a function that sums the squares from 0 through n.

Expected:
The sum should include n in the squared terms.

Actual:
The current result does not include n.

Code:
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total

Tried:
I checked the formula, but I did not check the exact values generated by the loop.
""".strip()

print(base_prompt)


My Python code does not work. Please help me debug it, but do not give me the solution or corrected code.

Problem:
I want to write a function that sums the squares from 0 through n.

Expected:
The sum should include n in the squared terms.

Actual:
The current result does not include n.

Code:
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total

Tried:
I checked the formula, but I did not check the exact values generated by the loop.


Send the base prompt to the model.

Look for this question: does the model guide the student, or does it jump straight to final code?


In [22]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
messages = [
    {"role": "user", "content": base_prompt}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=512,
    temperature=0.1,
)

base_output = response["choices"][0]["message"]["content"]
print(base_output)


You're on the right track with your function, but there's a small issue with the loop range. The `np.arange` function is not necessary here since you're already iterating over a range of integers from 0 to `n-1`. The loop should iterate from 0 to `n` inclusive. Here’s your function with the correct range:

```python
def sum_squares(n):
    total = 0
    for i in range(0, n + 1):
        total = total + i ** 2
    return total
```

You can test this function with different values of `n` to ensure it works as expected. For example:

- `sum_squares(0)` should return `0`.
- `sum_squares(1)` should return `1`.
- `sum_squares(2)` should return `5`.
- `sum_squares(3)` should return `14`.
- `sum_squares(4)` should return `30`.

Make sure to test these values to verify that the function is summing the squares correctly from 0 to `n`.


### Checkpoint 1

Did the base output follow the request?

Check two things:

1. Did it guide the student to inspect evidence first?
2. Did it give a solution or corrected code even though the prompt asked it not to?

*Double click to type your answer here*


## Step 4: Write the model instructions

Now JSON becomes useful.

| Output style | Good for | Problem |
|---|---|---|
| 💬 Plain text | Reading an explanation | Hard for code to reuse |
| 🧾 JSON | Pulling out specific fields | More rigid to write |

We use JSON here because the notebook needs the same fields every time:

| Field | What the student looks for |
|---|---|
| `likely_cause` | What might be wrong? |
| `next_check` | What should we inspect next? |
| `hint` | How can we think about it without final code? |
| `verify` | How should we test the fix? |

JSON does not make the model automatically correct. It makes the answer easier to inspect, compare, and reuse in code.


In [23]:
# RUN THIS CELL
instructions = """
You are a debugging coach for beginning Python students.
Do not give final corrected code.
Use only the student's report and any check result provided by the notebook.

Return valid JSON with these exact fields:
status, known, need, likely_cause, next_check, tool_name, hint, verify, reflect.

If Problem, Expected, Actual, Code, and Tried are present, use status = "ready" and need = "none".
Use status = "need" only if one of those sections is missing.

Use tool_name = "run_check" if one small evidence check would help.
Make next_check concrete. Prefer checking exact values over giving a fix.
Use tool_name = "" if no check is needed.
Do not write code for the tool.
The notebook has only one allowed tool name: run_check.
The notebook will choose the prepared classroom check.
""".strip()

json_mode = {"type": "json_object"}



## Step 5: Choose structured input

Now we give the model a structured bug report.

Each case uses the same five parts:

| Part | Meaning |
|---|---|
| Problem | What the student wanted to do |
| Expected | What should have happened |
| Actual | What happened instead |
| Code | The code being debugged |
| Tried | What the student already checked |

This makes the model less likely to guess and makes the answers easier to compare.

Each case also has one prepared classroom check for the tool step.


In [24]:
# JUST RUN THIS CELL
test_cases = [
    {
        "name": "Loop boundary",
        "problem": "I want to write a function that sums squares from 0 through n.",
        "expected": "I expect the sum to include n in the squared terms.",
        "actual": "I see that the current result does not include n.",
        "code": """def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total""",
        "tried": "I checked the formula, but I did not check the exact values generated by the loop.",
        "check_code": "print(np.arange(0, 3, 1))",
    },
    {
        "name": "Table column",
        "problem": "I want to create a num_tags column with one value per row.",
        "expected": "I expect each row to count the tags in that row.",
        "actual": "I see that the code counts the word tags instead of the column values.",
        "code": """def count_tags(string):
    return len(string.split(" "))

videos = Table().with_columns(
    "creator", make_array("a", "b", "c"),
    "tags", make_array("#data #python", "#study", "#fun #data")
)

videos = videos.with_column("num_tags", count_tags("tags"))""",
        "tried": "I tried using with_column because I want to add a new column.",
        "check_code": """def count_tags(string):
    return len(string.split(" "))

print(count_tags("tags"))""",
    },
    {
        "name": "Simulation deck",
        "problem": "I want to simulate a game with cards 1, 2, 3, and 4.",
        "expected": "I expect each game to draw from a four-card deck.",
        "actual": "I see that the deck expression leaves out the final card.",
        "code": """def simulate_game(n):
    wins = 0
    for i in np.arange(n):
        deck = np.arange(1, 4)
        cards = np.random.choice(deck, 2, replace=False)
        if cards.item(0) > cards.item(1):
            wins = wins + 1
    return wins / n""",
        "tried": "I inspected the random choice line, but I did not inspect the deck values.",
        "check_code": "print(np.arange(1, 4))",
    },
]

case_names = []
for case in test_cases:
    case_names.append(case["name"])

case_dropdown = widgets.Dropdown(
    options=case_names,
    value=case_names[0],
    description="Problem:",
)

display(case_dropdown)


Dropdown(description='Problem:', options=('Loop boundary', 'Table column', 'Simulation deck'), value='Loop bou…

Build the student bug report.

This turns the selected case into the structured input that the model will read.


In [25]:
# RUN THIS CELL AFTER CHOOSING A PROBLEM
selected_case = test_cases[0]
for case in test_cases:
    if case["name"] == case_dropdown.value:
        selected_case = case

prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
{selected_case["problem"]}

Expected:
{selected_case["expected"]}

Actual:
{selected_case["actual"]}

Code:
```python
{selected_case["code"]}
```

Tried:
{selected_case["tried"]}
""".strip()

print("Selected case:", selected_case["name"])
print(prompt)


Selected case: Loop boundary
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
I want to write a function that sums squares from 0 through n.

Expected:
I expect the sum to include n in the squared terms.

Actual:
I see that the current result does not include n.

Code:
```python
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total
```

Tried:
I checked the formula, but I did not check the exact values generated by the loop.


Ask Qwen2.5 for a guided JSON response.

This time, the response should be structured. Look for `likely_cause`, `next_check`, `hint`, and `verify`.


In [26]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": prompt}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=600,
    temperature=0.1,
    response_format=json_mode,
)

raw = response["choices"][0]["message"]["content"]
result = json.loads(raw)
print(json.dumps(result, indent=2))


{
  "status": "ready",
  "known": "The function is missing the import statement for numpy.",
  "need": "run_check",
  "likely_cause": "The function is not using the correct range of numbers to sum.",
  "next_check": "Verify that the loop is iterating from 0 to n (inclusive).",
  "tool_name": "",
  "hint": "Ensure the loop includes n in the range.",
  "verify": "Check the loop range and the values of i in the loop.",
  "reflect": "Remember, the loop should include n in the squared terms."
}


### Checkpoint 2

Compare the base output from Step 3 with the JSON output from Step 5.

Was JSON useful here?

Check four fields:

1. `likely_cause`: Does it name a possible cause?
2. `next_check`: Does it suggest something to inspect?
3. `hint`: Does it guide without giving the final code?
4. `verify`: Does it ask the student to test the fix?

Then decide: did JSON make the answer easier to check, or just more complicated?

*Double click to type your answer here*


## Step 6: Run a diagnostic check

Now tool use becomes useful.

The model can suggest a check, but it does not run Python. The notebook runs Python.

| Step | Who does it? | What happens? |
|---|---|---|
| 1 | 🤖 Model | Suggests `run_check` |
| 2 | 📓 Notebook | Decides whether the tool is allowed |
| 3 | 🐍 Python | Runs the diagnostic check |
| 4 | 👩‍🎓 Student | Decides whether the evidence supports the fix |

This is the key idea:

> The model suggests. Python produces evidence. The student verifies.

This is a beginner tool loop, not native API tool calling yet. The real API version appears in Optional Extension C.


In [27]:
# RUN THIS CELL
allowed_tool_name = "run_check"
requested_tool_name = result.get("tool_name", "")

# The model only chose a tool name.
# The notebook still decides what code can run.
print("Requested tool:", requested_tool_name)

if requested_tool_name != allowed_tool_name:
    check_result = None
    print("No allowed tool was requested.")
else:
    print("Allowed tool request found.")
    print("The notebook will run this prepared classroom check:")
    print(selected_case["check_code"])

    check_program = """
import numpy as np
""" + "\n" + selected_case["check_code"]

    completed = subprocess.run(
        [sys.executable, "-I", "-c", check_program],
        capture_output=True,
        text=True,
        timeout=3,
    )

    check_result = {
        "ok": completed.returncode == 0,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }

    print(json.dumps(check_result, indent=2))


Requested tool: 
No allowed tool was requested.


Send the check result back.

The check result is evidence. The model uses it to update the diagnosis.

The student still verifies the final fix.


In [28]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
if check_result is None:
    updated = None
    print("No tool result to send back.")
else:
    followup_prompt = f"""
I ran the requested check.

Previous JSON:
{json.dumps(result, indent=2)}

Check result:
{json.dumps(check_result, indent=2)}

Use the check result as evidence. Update likely_cause, next_check, hint, verify, and reflect. Return the same JSON fields.
""".strip()

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": followup_prompt}
    ]

    response = model.create_chat_completion(
        messages=messages,
        max_tokens=600,
        temperature=0.1,
        response_format=json_mode,
    )

    updated_raw = response["choices"][0]["message"]["content"]
    updated = json.loads(updated_raw)
    print(json.dumps(updated, indent=2))


No tool result to send back.


### Checkpoint 3

Was the tool step useful here?

Answer three questions:

1. What evidence did Python show?
2. Could the model know that evidence without running code?
3. Who still decides whether the final fix is correct?

*Double click to type your answer here*


## Step 7: Compare example cases

Run the same prompt pattern on every case and compare the next checks.


In [29]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
rows = []

for case in test_cases:
    case_prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
{case["problem"]}

Expected:
{case["expected"]}

Actual:
{case["actual"]}

Code:
```python
{case["code"]}
```

Tried:
{case["tried"]}
""".strip()

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": case_prompt}
    ]

    response = model.create_chat_completion(
        messages=messages,
        max_tokens=600,
        temperature=0.1,
        response_format=json_mode,
    )

    case_raw = response["choices"][0]["message"]["content"]
    case_result = json.loads(case_raw)

    rows.append({
        "Case": case["name"],
        "Status": case_result.get("status", ""),
        "Likely Cause": case_result.get("likely_cause", ""),
        "Next Check": case_result.get("next_check", ""),
        "Tool": case_result.get("tool_name", ""),
    })

results_table = pd.DataFrame(rows)
results_table


,Case,Status,Likely Cause,Next Check,Tool
0,Loop boundary,ready,The function is not using the correct range of...,Verify that the loop is iterating from 0 to n ...,
1,Table column,ready,The function count_tags is not being applied t...,Check the value of `videos['tags']` to see if ...,
2,Simulation deck,ready,The deck is missing the number 4.,Run the code and print the deck to inspect its...,


### Checkpoint 4

Which next check is clearest? What makes it useful for a beginner?

*Double click to type your answer here*


## Optional Extension Setup: OpenRouter API

The rest of the notebook is optional and uses OpenRouter through the OpenAI Python client.

The API model is **Qwen3.6 27B**.

Before running the setup cell, check that your `.env` file has a line like this:

`OPENROUTER_API_KEY=sk-or-v1-...`


In [30]:
# RUN THIS CELL
if "load_dotenv" in globals():
    load_dotenv()
else:
    print("python-dotenv is not installed. Paste your API key below if needed.")

api_model = "qwen/qwen3.6-27b"
api_key = os.getenv("OPENROUTER_API_KEY")

# Uncomment the next line and paste your key if load_dotenv did not find it.
# OPENROUTER_API_KEY = "sk-or-v1-..."  # Replace with your actual OpenRouter API key if needed.
# api_key = OPENROUTER_API_KEY

if "OpenAI" not in globals():
    api = None
    print("The openai package is not installed, so the optional API section cannot run yet.")
elif api_key:
    api = OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("OpenRouter client is ready.")
else:
    api = None
    print("OpenRouter API key was not found.")


OpenRouter client is ready.


## Optional Extension A: Thinking Mode

Thinking Mode asks the API model to spend extra reasoning effort.

This section compares the answer, input tokens, output tokens, total tokens, and time.

If `Text Returned` is `no`, the model used tokens but did not return final text. That is also a useful result to notice.


In [31]:
# RUN THIS CELL
thinking_rows = []

if api is None:
    print("OpenRouter call skipped because no API key was loaded.")
else:
    api_modes = ["base", "thinking"]

    for api_mode in api_modes:
        start_time = time.time()
        extra_body = {}

        if api_mode == "thinking":
            extra_body = {
                "reasoning": {
                    "max_tokens": 300,
                    "exclude": True
                }
            }

        api_response = api.chat.completions.create(
            model=api_model,
            messages=[
                {"role": "system", "content": instructions},
                {"role": "user", "content": prompt}
            ],
            response_format=json_mode,
            temperature=0.1,
            max_completion_tokens=900,
            extra_body=extra_body,
        )

        elapsed_time = time.time() - start_time
        api_raw = api_response.choices[0].message.content
        token_usage = api_response.usage

        if api_raw is None:
            api_result = {}
            text_returned = "no"
        else:
            api_result = json.loads(api_raw)
            text_returned = "yes"

        thinking_rows.append({
            "Mode": api_mode,
            "Input Tokens": token_usage.prompt_tokens,
            "Output Tokens": token_usage.completion_tokens,
            "Total Tokens": token_usage.total_tokens,
            "Seconds": round(elapsed_time, 2),
            "Text Returned": text_returned,
            "Status": api_result.get("status", ""),
            "Next Check": api_result.get("next_check", ""),
        })

thinking_table = pd.DataFrame(thinking_rows)
display(thinking_table)


,Mode,Input Tokens,Output Tokens,Total Tokens,Seconds,Text Returned,Status,Next Check
0,base,357,1937,2294,27.03,yes,ready,Run a check that prints the exact sequence of ...
1,thinking,357,573,930,9.44,yes,ready,Print the exact list of values produced by np....


### Extension Checkpoint A

Did Thinking Mode change the token count, time, or returned text?

Look at these columns:

1. `Input Tokens`
2. `Output Tokens`
3. `Total Tokens`
4. `Seconds`
5. `Text Returned`

*Double click to type your answer here*


## Optional Extension B: Multimodal Input

A multimodal model can read more than one kind of input.

This extension sends `debugging_screenshot.png` to OpenRouter and asks the model to extract the bug report fields.


In [32]:
# RUN THIS CELL
screenshot_path = "/Users/seohachoi/small_models/debugging_screenshot.png"
print("Screenshot path:", screenshot_path)
print("File exists:", os.path.exists(screenshot_path))


Screenshot path: /Users/seohachoi/small_models/debugging_screenshot.png
File exists: True


Extract fields from the image.

The image model should extract the bug report. It should not solve the bug yet.


In [33]:
# RUN THIS CELL
if api is None:
    screenshot_fields = None
    print("OpenRouter call skipped because no API key was loaded.")
elif not os.path.exists(screenshot_path):
    screenshot_fields = None
    print("Screenshot file was not found.")
else:
    image_file = open(screenshot_path, "rb")
    image_bytes = image_file.read()
    image_file.close()

    encoded_image = base64.b64encode(image_bytes).decode("utf-8")
    image_data_url = "data:image/png;base64," + encoded_image

    extraction_prompt = """
Read this debugging report image.
Extract exactly these lowercase keys: problem, expected, actual, code, tried.
Return valid JSON only.
Do not solve the bug.
""".strip()

    screenshot_response = api.chat.completions.create(
        model=api_model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": extraction_prompt},
                    {"type": "image_url", "image_url": {"url": image_data_url}}
                ]
            }
        ],
        response_format=json_mode,
        temperature=0.1,
        max_completion_tokens=600,
    )

    screenshot_raw = screenshot_response.choices[0].message.content

    if screenshot_raw is None:
        screenshot_fields = None
        print("No text was returned from the image model.")
    else:
        screenshot_fields = json.loads(screenshot_raw)
        print(json.dumps(screenshot_fields, indent=2))


{
  "problem": "I want to write a function that sums the squares from 0 through n.",
  "expected": "The sum should include n in the squared terms. For n = 3, the answer should include 3 squared.",
  "actual": "The current result does not include n. It stops one number too early.",
  "code": "def sum_squares(n):\n    total = 0\n    for i in np.arange(0, n, 1):\n        total = total + i ** 2\n    return total",
  "tried": "I checked the formula, but I did not check the exact values generated by the loop."
}


Send the extracted fields through the debugging workflow.

The image has become structured text, so the same workflow can use it.


In [34]:
# RUN THIS CELL
if api is None:
    screenshot_debug_result = None
    print("OpenRouter call skipped because no API key was loaded.")
elif screenshot_fields is None:
    screenshot_debug_result = None
    print("No screenshot fields are available.")
else:
    screenshot_prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
{screenshot_fields["problem"]}

Expected:
{screenshot_fields["expected"]}

Actual:
{screenshot_fields["actual"]}

Code:
```python
{screenshot_fields["code"]}
```

Tried:
{screenshot_fields["tried"]}
""".strip()

    screenshot_debug_response = api.chat.completions.create(
        model=api_model,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": screenshot_prompt}
        ],
        response_format=json_mode,
        temperature=0.1,
        max_completion_tokens=600,
    )

    screenshot_debug_raw = screenshot_debug_response.choices[0].message.content

    if screenshot_debug_raw is None:
        screenshot_debug_result = None
        print("No text was returned from the debugging model.")
    else:
        screenshot_debug_result = json.loads(screenshot_debug_raw)
        print(json.dumps(screenshot_debug_result, indent=2))


No text was returned from the debugging model.


### Extension Checkpoint B

Why should the image model extract fields before the debugging assistant gives advice?

*Double click to type your answer here*


## Optional Extension C: API Tool Calling

This extension shows real API tool calling with OpenRouter.

The same rule still applies: the model does not run the tool by itself.

| Step | What happens |
|---|---|
| 1 | Send a tool description to the model |
| 2 | Check whether the model returned `tool_calls` |
| 3 | Run the allowed prepared check in Python |
| 4 | Send the tool result back to the model |

Tool calling is useful when an AI assistant needs outside evidence:

| Need | Example tool evidence |
|---|---|
| Code result | Run a Python check |
| Data result | Query a table or database |
| File content | Read a file |
| API response | Ask a service for current information |

The safety habit is the same: **the model asks, the program checks, Python runs.**


In [35]:
# RUN THIS CELL
debugging_tools = [
    {
        "type": "function",
        "function": {
            "name": "run_check",
            "description": "Run one prepared classroom debugging check.",
            "parameters": {
                "type": "object",
                "properties": {
                    "case_name": {
                        "type": "string",
                        "description": "The selected debugging case name."
                    }
                },
                "required": ["case_name"]
            }
        }
    }
]

tool_prompt = f"""
Use the run_check tool before you give debugging guidance.
Do not give final corrected code.

Selected case: {selected_case["name"]}

Problem:
{selected_case["problem"]}

Expected:
{selected_case["expected"]}

Actual:
{selected_case["actual"]}

Code:
```python
{selected_case["code"]}
```

Tried:
{selected_case["tried"]}
""".strip()

if api is None:
    first_tool_message = None
    tool_call_list = []
    print("OpenRouter call skipped because no API key was loaded.")
else:
    tool_messages = [
        {"role": "system", "content": "You are a debugging coach. Use tools when a check would help."},
        {"role": "user", "content": tool_prompt}
    ]

    first_tool_response = api.chat.completions.create(
        model=api_model,
        messages=tool_messages,
        tools=debugging_tools,
        tool_choice="auto",
        temperature=0.1,
        max_completion_tokens=600,
    )

    first_tool_message = first_tool_response.choices[0].message
    tool_call_list = first_tool_message.tool_calls

    print("Finish reason:", first_tool_response.choices[0].finish_reason)
    print("Tool calls:", tool_call_list)
    if not tool_call_list:
        print("The model did not request a tool call.")
        print(first_tool_message.content)


Finish reason: tool_calls
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_2a5daa8b8c6c4f7a80bb06f7', function=Function(arguments='{"case_name": "Loop boundary"}', name='run_check'), type='function', index=0)]


Run the API tool call if one was requested.

This cell checks both the tool name and the `case_name` argument before running the prepared check.


In [36]:
# RUN THIS CELL
if api is None:
    final_tool_answer = None
    print("OpenRouter call skipped because no API key was loaded.")
elif not tool_call_list:
    final_tool_answer = None
    print("No tool call is available to run.")
else:
    assistant_message = {
        "role": "assistant",
        "content": first_tool_message.content,
        "tool_calls": []
    }

    for tool_call in tool_call_list:
        assistant_message["tool_calls"].append({
            "id": tool_call.id,
            "type": "function",
            "function": {
                "name": tool_call.function.name,
                "arguments": tool_call.function.arguments
            }
        })

    tool_messages.append(assistant_message)

    for tool_call in tool_call_list:
        tool_name = tool_call.function.name
        tool_arguments = json.loads(tool_call.function.arguments)

        print("Tool name:", tool_name)
        print("Tool arguments:", tool_arguments)

        requested_case_name = tool_arguments.get("case_name", "")

        if tool_name == "run_check" and requested_case_name == selected_case["name"]:
            check_program = """
import numpy as np
""" + "\n" + selected_case["check_code"]

            completed = subprocess.run(
                [sys.executable, "-I", "-c", check_program],
                capture_output=True,
                text=True,
                timeout=3,
            )

            api_tool_result = {
                "ok": completed.returncode == 0,
                "stdout": completed.stdout,
                "stderr": completed.stderr,
            }
        elif tool_name == "run_check":
            api_tool_result = {"error": "The requested case name does not match the selected case."}
        else:
            api_tool_result = {"error": "This tool is not allowed."}

        tool_messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(api_tool_result),
        })

    final_tool_response = api.chat.completions.create(
        model=api_model,
        messages=tool_messages,
        tools=debugging_tools,
        temperature=0.1,
        max_completion_tokens=600,
    )

    final_tool_answer = final_tool_response.choices[0].message.content
    print(final_tool_answer)


Tool name: run_check
Tool arguments: {'case_name': 'Loop boundary'}
None


### Extension Checkpoint C

In API tool calling, who controls each part?

1. Who may request the tool call?
2. Who checks the tool name and arguments?
3. Who runs the Python code?

*Double click to type your answer here*


## Summary

You built a local debugging assistant with Qwen2.5 3B Q4.

The assistant supports this habit:

1. Observe the problem.
2. Name a likely cause.
3. Run a diagnostic check.
4. Verify the fix.
5. Reflect on the habit.

You also practiced two levels of tool use:

1. A beginner tool loop in the local notebook.
2. Real API tool calling in the optional OpenRouter extension.

The main lesson is that the model should support structured debugging practice. It should not replace student verification.
